# 00 — Prepare Decoder Inputs

**Purpose:** Turn the beta-series catalog into *decoder-ready* tables — one table
per decoding question. This is the bookkeeping notebook for the whole decoding
series. No machine learning happens here yet. The goal is to get the **inputs**
into a clean, well-understood shape so that every later notebook can start from a
single CSV.

This notebook follows the same style as the beta-series notebooks: read the
markdown carefully, then fill in the `TODO`s yourself. The code cells are
deliberately incomplete — the point is for *you* to write the pandas.

---

## Notebook overview

You already produced 64 single-trial beta maps for `sub-097` (32 trials × 2 runs)
and catalogued them in:

```
C:\ManzaRotation\Analysis\outputs\sub-097\decoding\tables\sub-097_catalog.csv
```

That catalog has one row per beta map, with columns:

| column | meaning |
|---|---|
| `subject` | always `sub-097` here |
| `task` | which run the trial came from (`modulate1` or `modulate2`) |
| `condition` | the emotion condition (`pos_val`, `neg_val`, `pos_aro`, `neg_aro`) |
| `trial_num` | the trial index within that condition+run |
| `path` | absolute path to the trial's beta `.nii.gz` |

By the end of this notebook you will have written four new tables, each one
describing a single decoding question:

```
decoder_posval_vs_negval.csv      # positive vs negative VALENCE
decoder_posaro_vs_negaro.csv      # positive vs negative AROUSAL
decoder_valence_vs_arousal.csv    # valence trials vs arousal trials
decoder_4class.csv                # all four conditions at once
```

## What you are learning

- What a **decoder input table** is and why each row is a single trial.
- The four words that describe *every* supervised-learning problem:
  **samples**, **features**, **labels (`y`)**, and **groups**.
- Why labels are stored *separately* from the images.
- Why **groups** are needed for honest cross-validation.
- The difference between **binary** and **multiclass** decoding.
- How the *scientific question* you ask determines the labels you build.

## Why this matters scientifically

A decoder can only answer the question you encode in its labels. If you label
trials by valence, a high accuracy means "valence is represented in the brain
pattern." If you relabel the *same* images by arousal, you are now asking a
completely different question. Getting the input tables right is therefore not
clerical busywork — **the labels *are* the hypothesis.** Sloppy tables silently
change what your "result" means.

---
## Section 1 — Configuration and Load the Catalog

As in the beta-series notebooks, define everything from `subject` so the paths
stay general. The catalog lives in the `decoding/tables` output folder.

Reading a CSV with pandas:

```python
catalog = pd.read_csv(catalog_path)
```

**TODO:**
- [ ] Set `subject`.
- [ ] Build `decoding_dir` and `tables_dir` from `project_dir`.
- [ ] Read `sub-097_catalog.csv` into a DataFrame called `catalog`.
- [ ] Print `catalog.shape` (expect `(64, 5)`).

In [40]:
from pathlib import Path
import pandas as pd

subject = "sub-097"

project_dir  = Path(r"C:\ManzaRotation")
decoding_dir = project_dir / "Analysis" / "outputs" / subject / "decoding"
tables_dir   = decoding_dir / "tables"

catalog_path = tables_dir / f"{subject}_catalog.csv"

# TODO: read the catalog CSV into a DataFrame called `catalog`
catalog = pd.read_csv(catalog_path)
# TODO: print catalog.shape  -> expect (64, 5)
catalog.shape
# TODO: display the first few rows with catalog.head()
catalog.head()


,subject,task,condition,trial_num,path
0,sub-097,modulate1,neg_aro,1,C:\ManzaRotation\Analysis\outputs\sub-097\beta...
1,sub-097,modulate1,neg_aro,2,C:\ManzaRotation\Analysis\outputs\sub-097\beta...
2,sub-097,modulate1,neg_aro,3,C:\ManzaRotation\Analysis\outputs\sub-097\beta...
3,sub-097,modulate1,neg_aro,4,C:\ManzaRotation\Analysis\outputs\sub-097\beta...
4,sub-097,modulate1,neg_aro,5,C:\ManzaRotation\Analysis\outputs\sub-097\beta...


---
## Section 2 — Inspect the Counts

Before building anything, *look* at what you have. Two pandas tools do almost all
of the work when inspecting categorical columns:

- `series.value_counts()` — counts how many rows have each value in one column.
- `df.groupby([...]).size()` — counts rows for each combination of columns.

For example:

```python
catalog["condition"].value_counts()          # how many trials per condition?
catalog.groupby(["task", "condition"]).size() # per run AND condition
```

**Sanity check you are aiming for:** 4 conditions × 8 trials × 2 runs = 64 rows,
so each condition should have 16 trials total (8 per run).

**TODO:**
- [ ] Print `value_counts()` for the `condition` column.
- [ ] Print `value_counts()` for the `task` column.
- [ ] Print the `groupby(["task", "condition"]).size()` table.
- [ ] Confirm the numbers match the sanity check above.

In [11]:
# TODO: how many trials of each condition?
print(catalog["condition"].value_counts())

# TODO: how many trials per run?
print(catalog["task"].value_counts())

# TODO: per run AND condition (should be 8 everywhere)
print(catalog.groupby(["task", "condition"]).size())

condition
neg_aro    16
neg_val    16
pos_aro    16
pos_val    16
Name: count, dtype: int64
task
modulate1    32
modulate2    32
Name: count, dtype: int64
task       condition
modulate1  neg_aro      8
           neg_val      8
           pos_aro      8
           pos_val      8
modulate2  neg_aro      8
           neg_val      8
           pos_aro      8
           pos_val      8
dtype: int64


---
## Section 3 — The Four Words: samples, features, labels, groups

Every supervised-learning problem — including fMRI decoding — is described by
four things. Learn these words now; they appear in every later notebook.

**Samples** — the *things you classify*. Here, **one sample = one trial's beta
map**. You have 64 samples. In scikit-learn the samples are the rows of `X`.

**Features** — the *numbers describing each sample*. Here the features are the
**voxel values** inside a brain mask. A beta map has tens of thousands of voxels,
so each sample is a very long vector. `X` has shape `(n_samples, n_features)` —
think *(trials, voxels)*. (Nilearn will build this matrix for you from the
images; you rarely materialise it by hand.)

**Labels (`y`)** — the *answer* for each sample: the category you want the
decoder to predict. `y` is a 1-D array with one entry per sample, e.g.
`["pos_val", "pos_val", "neg_val", ...]`.

**Groups** — a label that says *which trials are not independent of each other*.
Trials from the same scanner run share noise (same head position, same drift,
same day). For honest evaluation you must never train and test on the same run.
The `groups` array (here, the `task` column) lets cross-validation keep whole
runs together. More on this in Section 4.

A useful mental picture of what the decoder eventually sees:

```
            feature_1  feature_2  ...  feature_50000     y          group
trial 1   [  0.21      -0.04     ...     0.88     ]   pos_val    modulate1
trial 2   [ -0.10       0.33     ...     0.05     ]   pos_val    modulate1
...
trial 64  [  0.42       0.19     ...    -0.27     ]   neg_aro    modulate2
```

The images supply the left block (features). Your **table** supplies the right
two columns (`y` and `group`). That is the whole reason this notebook exists.

---
## Section 4 — Why Labels and Groups Live in a Table, Not in the Images

A beta map is just a brain volume of numbers — it does **not** carry its own
condition label or run identity. Those facts live *outside* the image, in your
catalog. Keeping them in a table (rather than, say, only in the filename) has
three big payoffs:

1. **Separation of concerns.** Images are features; the table is metadata. You
   can relabel the *same* images for a different question (Section 7) without
   touching a single `.nii.gz`.
2. **Alignment.** As long as you build `imgs`, `y`, and `groups` by iterating the
   **same DataFrame in the same order**, sample *i* of `X` always matches entry
   *i* of `y` and entry *i* of `groups`. Misalignment here is the single most
   common decoding bug.
3. **Honest cross-validation.** With only two runs, the natural validation scheme
   is **leave-one-run-out**: train on `modulate1`, test on `modulate2`, then swap.
   The `groups` column is what makes that possible.

**TODO (no code — answer in the markdown cell below):**
- [ ] In your own words, what would go wrong if you shuffled `y` but not `imgs`?
- [ ] Why is it *not* safe to put trials from the same run in both train and test?

*Your answers:*

> (write here)

---
## Section 5 — Binary vs Multiclass Decoding

The number of distinct values in `y` decides what *kind* of problem you have.

- **Binary** — exactly two classes. The decoder draws one boundary. Chance
  performance for a balanced binary problem is **50%**.
- **Multiclass** — three or more classes. The decoder must separate several
  categories at once; this is harder, and chance is **1 / (number of classes)**.
  For the 4-class problem here, chance is **25%**.

"Chance level" is the accuracy you would expect from a decoder that has learned
nothing — it is the bar your real accuracy must clear to be meaningful. Always
write it down *before* you look at your score, so you cannot fool yourself.

This series builds three binary tables and one multiclass table:

| table | classes | chance |
|---|---|---|
| `decoder_posval_vs_negval` | pos_val, neg_val | 50% |
| `decoder_posaro_vs_negaro` | pos_aro, neg_aro | 50% |
| `decoder_valence_vs_arousal` | valence, arousal | 50% |
| `decoder_4class` | all four conditions | 25% |

---
## Section 6 — Build the First Subset: `pos_val` vs `neg_val`

Now the pandas. The first decoding question is *positive vs negative valence*, so
you want only the rows whose `condition` is `pos_val` or `neg_val`.

**Boolean filtering** is the tool. `series.isin([...])` returns a True/False mask
that you use to select rows:

```python
mask   = catalog["condition"].isin(["pos_val", "neg_val"])
subset = catalog[mask].copy()   # .copy() avoids the SettingWithCopyWarning later
```

Then add the two columns the decoder cares about. Storing them explicitly (even
when `label` just duplicates `condition`) keeps every decoder table in the *same
shape*, which makes notebooks 01–04 interchangeable:

```python
subset["label"]  = subset["condition"]   # what we predict (y)
subset["groups"] = subset["task"]        # run identity for CV
```

**TODO:**
- [ ] Build the boolean mask for `pos_val`/`neg_val` and select those rows.
- [ ] Add a `label` column equal to `condition`.
- [ ] Add a `groups` column equal to `task`.
- [ ] Print `subset["label"].value_counts()` — expect 16 and 16.
- [ ] Print `subset.groupby(["groups", "label"]).size()` to confirm balance per run.

In [22]:
# TODO 1: Look at the condition column
# Hint: catalog["condition"].value_counts()

# TODO 2: Make a list of the two conditions we want
# Hint: target_conditions = ["pos_val", "neg_val"]
target_conditions = ["pos_val", "neg_val"]
# TODO 3: Make a True/False mask for rows with those conditions
# Hint: catalog["condition"].isin(target_conditions)
mask = catalog["condition"].isin(target_conditions)
# TODO 4: Use the mask to create a new filtered DataFrame
catalog_copy = catalog.copy()
filtered_catalog = catalog_copy[mask]

# TODO 5: Check the shape of the new DataFrame
# Expected: 32 rows
# TODO 6: Add a new column called label
# Hint: label should equal condition
filtered_catalog["label"] = filtered_catalog["condition"]
# TODO 7: Add a new column called groups
# Hint: groups should equal task/run
filtered_catalog.shape
# TODO 8: Print the first few rows
# Hint: use .head()
print(filtered_catalog)
# TODO 9: Check overall label counts
# Expected: pos_val = 16, neg_val = 16

# TODO 10: Check counts per run and label
# Expected: 8 of each label in modulate1 and 8 of each in modulate2



    subject       task condition  trial_num  \
8   sub-097  modulate1   neg_val          1   
9   sub-097  modulate1   neg_val          2   
10  sub-097  modulate1   neg_val          3   
11  sub-097  modulate1   neg_val          4   
12  sub-097  modulate1   neg_val          5   
13  sub-097  modulate1   neg_val          6   
14  sub-097  modulate1   neg_val          7   
15  sub-097  modulate1   neg_val          8   
24  sub-097  modulate1   pos_val          1   
25  sub-097  modulate1   pos_val          2   
26  sub-097  modulate1   pos_val          3   
27  sub-097  modulate1   pos_val          4   
28  sub-097  modulate1   pos_val          5   
29  sub-097  modulate1   pos_val          6   
30  sub-097  modulate1   pos_val          7   
31  sub-097  modulate1   pos_val          8   
40  sub-097  modulate2   neg_val          1   
41  sub-097  modulate2   neg_val          2   
42  sub-097  modulate2   neg_val          3   
43  sub-097  modulate2   neg_val          4   
44  sub-097  

**Sanity checks to insist on before saving any decoder table:**

- The two classes are **balanced** (16 vs 16). Imbalance inflates accuracy and
  makes 50% the wrong chance level.
- Each class appears in **both** runs. If a class lived in only one run, a
  leave-one-run-out decoder could "decode" it by simply memorising the run.

---
## Section 7 — Save the Table (and a Reusable Helper)

You will repeat the Section 6 pattern four times, so it is worth wrapping it in a
small helper function. Writing the helper *yourself* is part of the exercise —
the skeleton and the docstring are given, the body is yours.

```python
def make_decoder_table(catalog, conditions, label_from="condition"):
    '''Return a decoder-ready subset of `catalog`.

    Parameters
    ----------
    conditions : list[str]
        Which `condition` values to keep.
    label_from : str
        Name of the column to copy into the new `label` column.
    '''
    # TODO: 1. filter rows whose condition is in `conditions`  (.isin)
    # TODO: 2. .copy() the result
    # TODO: 3. add `label`  = subset[label_from]
    # TODO: 4. add `groups` = subset["task"]
    # TODO: 5. return the subset
    ...
```

Saving a DataFrame to CSV (no index column):

```python
subset.to_csv(tables_dir / "decoder_posval_vs_negval.csv", index=False)
```

**TODO:**
- [ ] Implement `make_decoder_table`.
- [ ] Use it to rebuild the `pos_val` vs `neg_val` table.
- [ ] Save it as `decoder_posval_vs_negval.csv`.

In [24]:
def make_decoder_table(catalog, conditions, label_from="condition"):
    '''Return a decoder-ready subset of the catalog (see markdown above).'''
    # TODO: implement using .isin(), .copy(), and new label/groups columns
    mask = catalog["condition"].isin(conditions)
    filtered_catalog = catalog[mask].copy()
    filtered_catalog["label"] = filtered_catalog[label_from]
    filtered_catalog["groups"] = filtered_catalog["task"]

    return filtered_catalog

# TODO: 

posval_negval = make_decoder_table(catalog, ["pos_val", "neg_val"])
catalog_path = tables_dir / f"{subject}_posval_negval_catalog.csv"
posval_negval.to_csv(
    catalog_path,
    index=False
)


---
## Section 8 — The Arousal Table: `pos_aro` vs `neg_aro`

Exactly the same shape of problem, different conditions. This is the table
notebook 02 will consume.

**TODO:**
- [ ] Call `make_decoder_table` with `["pos_aro", "neg_aro"]`.
- [ ] Sanity-check balance (16 vs 16, present in both runs).
- [ ] Save as `decoder_posaro_vs_negaro.csv`.

In [25]:
# TODO: build and save the pos_aro vs neg_aro table

posaro_negaro = make_decoder_table(catalog, ["pos_aro", "neg_aro"])
catalog_path = tables_dir / f"{subject}_posaro_negaro_catalog.csv"
posaro_negaro.to_csv(
    catalog_path,
    index=False
)

---
## Section 9 — A Genuinely Different Question: Valence vs Arousal

The previous two tables split *within* a dimension (positive vs negative). This
table splits *between* dimensions: every valence trial (`pos_val` + `neg_val`)
against every arousal trial (`pos_aro` + `neg_aro`).

Notice that the **labels no longer match the conditions** — you are *collapsing*
four conditions into two new labels. This is your first taste of the idea that
"the label is the hypothesis": the same 64 images, regrouped, ask whether the
brain distinguishes *which dimension was being modulated* at all.

A clean way to derive the new label is a mapping dictionary plus `Series.map`:

```python
to_dimension = {
    "pos_val": "valence",
    "neg_val": "valence",
    "pos_aro": "arousal",
    "neg_aro": "arousal",
}
catalog["dimension"] = catalog["condition"].map(to_dimension)
```

**TODO:**
- [ ] Build the `to_dimension` mapping and add a `dimension` column to `catalog`.
- [ ] Build a decoder table whose `label` is `dimension` (all 64 rows are kept).
      *Hint:* you can pass `label_from="dimension"` and a `conditions` list of all
      four conditions, or filter differently — your choice.
- [ ] Check balance: 32 `valence` vs 32 `arousal`, present in both runs.
- [ ] Save as `decoder_valence_vs_arousal.csv`.

In [28]:
# TODO: add a `dimension` column via a mapping dict + .map()
to_dimension = {
    "pos_val": "valence",
    "neg_val": "valence",
    "pos_aro": "arousal",
    "neg_aro": "arousal"
}
catalog["dimension"] = catalog["condition"].map(to_dimension)
# TODO: build the valence-vs-arousal decoder table (label_from="dimension")
valence_vs_arousal = make_decoder_table(catalog, ["pos_aro", "neg_aro", "neg_val", "pos_val"], label_from="dimension")
# TODO: sanity-check balance, then save decoder_valence_vs_arousal.csv
print(valence_vs_arousal.shape)
print(valence_vs_arousal.head())
catalog_path = tables_dir / f"{subject}_valence_vs_arousal_catalog.csv"

valence_vs_arousal.to_csv(
    catalog_path,
    index=False
)



(64, 8)
   subject       task condition  trial_num  \
0  sub-097  modulate1   neg_aro          1   
1  sub-097  modulate1   neg_aro          2   
2  sub-097  modulate1   neg_aro          3   
3  sub-097  modulate1   neg_aro          4   
4  sub-097  modulate1   neg_aro          5   

                                                path dimension    label  \
0  C:\ManzaRotation\Analysis\outputs\sub-097\beta...   arousal  arousal   
1  C:\ManzaRotation\Analysis\outputs\sub-097\beta...   arousal  arousal   
2  C:\ManzaRotation\Analysis\outputs\sub-097\beta...   arousal  arousal   
3  C:\ManzaRotation\Analysis\outputs\sub-097\beta...   arousal  arousal   
4  C:\ManzaRotation\Analysis\outputs\sub-097\beta...   arousal  arousal   

      groups  
0  modulate1  
1  modulate1  
2  modulate1  
3  modulate1  
4  modulate1  


---
## Section 10 — The Multiclass Table: All Four Conditions

The last table keeps every trial and uses the original four conditions as the
label. Notebook 04 will use it for 4-class decoding (chance = 25%).

**TODO:**
- [ ] Build a decoder table containing all four conditions, `label` = `condition`.
- [ ] Confirm `label.value_counts()` shows 16 for each of the four classes.
- [ ] Save as `decoder_4class.csv`.

In [33]:
# TODO: build and save the 4-class decoder table

four_class = make_decoder_table(catalog, ["pos_aro", "neg_aro", "neg_val", "pos_val"])
catalog_path = tables_dir / f"{subject}_decoder_4class.csv"
four_class.to_csv(
    catalog_path,
    index=False
)
four_class["label"].value_counts()

label
neg_aro    16
neg_val    16
pos_aro    16
pos_val    16
Name: count, dtype: int64

---
## Section 11 — Final Check: List What You Built

End every preparation notebook by confirming the artifacts exist on disk.

**TODO:**
- [ ] Glob `tables_dir` for `decoder_*.csv` and print each filename.
- [ ] For each, re-read it and print its shape + `label.value_counts()`.
- [ ] Confirm: two binary tables of 32 rows, one binary table of 64 rows, one
      4-class table of 64 rows.

**What comes next:** Notebook 01 takes `decoder_posval_vs_negval.csv`, turns the
`path` column into images (`X`), the `label` column into `y`, the `groups` column
into cross-validation folds, and trains your first SVM decoder.

In [45]:
# TODO: list the decoder_*.csv files and print a one-line summary of each
sorted_dir =sorted(tables_dir.glob("*.csv"))

print(sorted_dir)

for path in sorted_dir:
    print(path)

[WindowsPath('C:/ManzaRotation/Analysis/outputs/sub-097/decoding/tables/sub-097_catalog.csv'), WindowsPath('C:/ManzaRotation/Analysis/outputs/sub-097/decoding/tables/sub-097_decoder_4class.csv'), WindowsPath('C:/ManzaRotation/Analysis/outputs/sub-097/decoding/tables/sub-097_posaro_negaro_catalog.csv'), WindowsPath('C:/ManzaRotation/Analysis/outputs/sub-097/decoding/tables/sub-097_posval_negval_catalog.csv'), WindowsPath('C:/ManzaRotation/Analysis/outputs/sub-097/decoding/tables/sub-097_valence_vs_arousal_catalog.csv')]
C:\ManzaRotation\Analysis\outputs\sub-097\decoding\tables\sub-097_catalog.csv
C:\ManzaRotation\Analysis\outputs\sub-097\decoding\tables\sub-097_decoder_4class.csv
C:\ManzaRotation\Analysis\outputs\sub-097\decoding\tables\sub-097_posaro_negaro_catalog.csv
C:\ManzaRotation\Analysis\outputs\sub-097\decoding\tables\sub-097_posval_negval_catalog.csv
C:\ManzaRotation\Analysis\outputs\sub-097\decoding\tables\sub-097_valence_vs_arousal_catalog.csv
